In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch



In [2]:
# Load the e5-large model and tokenizer
model_name = "intfloat/e5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)



tokenizer_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

2025-01-21 23:29:34.869821: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-01-21 23:29:34.906425: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-01-21 23:29:34.906460: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-01-21 23:29:34.907492: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-21 23:29:34.913389: I tensorflow/core/platform/cpu_feature_guar

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

In [3]:
# Function to generate embeddings
def generate_embedding(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :]  # CLS token embeddings
    return embeddings.squeeze().tolist()



In [4]:
# Load the input CSV file
input_file = "biobert_embedding_terms.csv"  # Replace with your input file path
output_file = "e5-large.csv"  # Replace with your output file path
df = pd.read_csv(input_file)

# Check if the required column exists
if "Tumor_Names" not in df.columns:
    raise ValueError("The input CSV file must contain a column labeled 'Tumor_Names'.")

# Generate embeddings for each tumor name


In [5]:
embeddings = []
for tumor_name in df["Tumor_Names"]:
    embedding = generate_embedding(tumor_name, tokenizer, model)
    embeddings.append(embedding)


In [6]:

# Convert embeddings to a DataFrame
embeddings_df = pd.DataFrame(embeddings)
embeddings_df.columns = [f"dim_{i+1}" for i in range(embeddings_df.shape[1])]



In [7]:
# Combine the tumor names with their embeddings
output_df = pd.concat([df["Tumor_Names"], embeddings_df], axis=1)

# Save to CSV
output_df.to_csv(output_file, index=False)
print(f"Embeddings saved to {output_file}")

Embeddings saved to e5-large.csv
